# Content-Based Personalization

**What this adds on top of our existing pipeline:**
- SQLite database to store users, searches, and clicks
- User tag-preference profiles built from click history
- Personalized scoring (user_tag_match signal in ranking)
- Cold start feed for new users (rule-based defaults)
- Homepage "Recommended for you" based on user profile

**Pipeline after this notebook:**
```
FAISS (retrieve 50) → Cross-Encoder (rank 20) → Weighted Score + Personalization (top 10)
```

---
## 1. Setup & Load Existing Pipeline

In [1]:
import numpy as np
import pandas as pd
import faiss
import sqlite3
import time
import os
import json
from datetime import datetime
from sentence_transformers import SentenceTransformer, CrossEncoder

print('Libraries imported!')

Libraries imported!


In [3]:
# ⚠️ UPDATE THIS PATH
DATA_DIR = '../Dataset_Cleaned'

# Load all components from previous notebooks
print('Loading pipeline components...')

index = faiss.read_index(os.path.join(DATA_DIR, 'faiss_index.bin'))
question_ids = np.load(os.path.join(DATA_DIR, 'question_ids.npy'))
questions_df = pd.read_parquet(os.path.join(DATA_DIR, 'questions_cleaned.parquet'))
answers_df = pd.read_parquet(os.path.join(DATA_DIR, 'answers_cleaned.parquet'))
print(f'  Data: {len(questions_df):,} questions, {len(answers_df):,} answers')

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f'  Models loaded')

# Build lookup structures
questions_dict = questions_df.set_index('id').to_dict('index')
answers_grouped = answers_df.sort_values('answer_rank').groupby('question_id')

print(f'\nAll pipeline components loaded!')

Loading pipeline components...
  Data: 498,644 questions, 1,199,383 answers


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Models loaded

All pipeline components loaded!


---
## 2. SQLite Database Setup

SQLite stores everything in a single `.db` file — no server needed. It deploys alongside our other files. Three tables: `users`, `search_history`, `click_history`.

In [5]:
DB_PATH = os.path.join(DATA_DIR, 'recsys_users.db')

def init_database():
    """Create the database and tables if they don't exist."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Users table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Search history — what the user searched
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS search_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            query TEXT NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    ''')
    
    # Click history — which results the user clicked
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS click_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            question_id INTEGER NOT NULL,
            question_tags TEXT NOT NULL,
            question_score INTEGER DEFAULT 0,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    ''')
    
    conn.commit()
    conn.close()
    print(f'Database initialized: {DB_PATH}')


init_database()

Database initialized: ../Dataset_Cleaned\recsys_users.db


---
## 3. User Management

In [8]:
def create_user(username):
    """Create a new user. Returns user_id."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        cursor.execute('INSERT INTO users (username) VALUES (?)', (username,))
        conn.commit()
        user_id = cursor.lastrowid
        print(f'  Created user: {username} (id: {user_id})')
        return user_id
    except sqlite3.IntegrityError:
        # User already exists
        cursor.execute('SELECT id FROM users WHERE username = ?', (username,))
        user_id = cursor.fetchone()[0]
        print(f'  User exists: {username} (id: {user_id})')
        return user_id
    finally:
        conn.close()


def get_user(username):
    """Get user_id by username. Returns None if not found."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT id FROM users WHERE username = ?', (username,))
    row = cursor.fetchone()
    conn.close()
    return row[0] if row else None


print('User management functions ready!')

User management functions ready!


---
## 4. Track Searches and Clicks

In [11]:
def log_search(user_id, query):
    """Record that a user made a search."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        'INSERT INTO search_history (user_id, query) VALUES (?, ?)',
        (user_id, query)
    )
    conn.commit()
    conn.close()


def log_click(user_id, question_id, question_tags, question_score):
    """Record that a user clicked on a search result."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        'INSERT INTO click_history (user_id, question_id, question_tags, question_score) VALUES (?, ?, ?, ?)',
        (user_id, question_id, question_tags, question_score)
    )
    conn.commit()
    conn.close()


def get_user_click_count(user_id):
    """How many clicks does this user have?"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT COUNT(*) FROM click_history WHERE user_id = ?', (user_id,))
    count = cursor.fetchone()[0]
    conn.close()
    return count


print('Tracking functions ready!')

Tracking functions ready!


---
## 5. Build User Tag Profile

This is the core of content-based personalization. We analyze what the user has clicked and build a tag-preference profile.

In [14]:
def build_user_tag_profile(user_id):
    """
    Build a tag-preference profile from the user's click history.
    
    Returns a dict of {tag: weight} where weights sum to 1.0.
    Recent clicks are weighted more heavily than older ones.
    
    Example:
        {'python': 0.55, 'pandas': 0.15, 'sorting': 0.10, ...}
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Get all clicks ordered by time (newest first)
    cursor.execute('''
        SELECT question_tags, timestamp 
        FROM click_history 
        WHERE user_id = ? 
        ORDER BY timestamp DESC
    ''', (user_id,))
    
    clicks = cursor.fetchall()
    conn.close()
    
    if not clicks:
        return {}  # No history — cold start
    
    # Count tags with recency weighting
    # Most recent click gets weight 1.0, older clicks decay
    tag_weights = {}
    total_clicks = len(clicks)
    
    for i, (tags_str, timestamp) in enumerate(clicks):
        # Recency weight: newest = 1.0, oldest = 0.3
        recency = 1.0 - (i / total_clicks) * 0.7
        
        tags = [t.strip() for t in tags_str.split(',') if t.strip()]
        for tag in tags:
            tag_weights[tag] = tag_weights.get(tag, 0) + recency
    
    # Normalize so weights sum to 1.0
    total = sum(tag_weights.values())
    if total > 0:
        tag_weights = {tag: round(w / total, 4) for tag, w in tag_weights.items()}
    
    # Sort by weight descending
    tag_weights = dict(sorted(tag_weights.items(), key=lambda x: -x[1]))
    
    return tag_weights


def compute_user_tag_match(question_tags, user_profile):
    """
    Compute how well a question's tags match the user's preferences.
    
    Returns a score between 0.0 (no match) and 1.0 (perfect match).
    """
    if not user_profile:
        return 0.0  # No profile — no personalization
    
    tags = [t.strip() for t in question_tags.split(',') if t.strip()]
    match_score = sum(user_profile.get(tag, 0) for tag in tags)
    
    # Cap at 1.0
    return min(1.0, match_score)


print('User profile functions ready!')

User profile functions ready!


---
## 6. Personalized Search Function

Same 3-stage pipeline as before, but now the weighted scoring includes a `user_tag_match` signal.

In [17]:
def get_question_details(q_id):
    """Get question + top answer details."""
    q = questions_dict.get(q_id)
    if q is None:
        return None
    top_answer = None
    if q_id in answers_grouped.groups:
        top_answer = answers_grouped.get_group(q_id).iloc[0]
    return {
        'question_id': q_id,
        'title': q['title'],
        'tags': q['tags'],
        'primary_tag': q['primary_tag'],
        'score': int(q['score']),
        'views': int(q['view_count']),
        'has_accepted': bool(q['has_accepted_answer']),
        'creation_date': q['creation_date'],
        'answer_body': top_answer['body'][:300] if top_answer is not None else '',
        'answer_score': int(top_answer['score']) if top_answer is not None else 0,
        'answer_accepted': bool(top_answer['is_accepted']) if top_answer is not None else False
    }


def compute_final_score(candidate, user_profile=None):
    """
    Weighted scoring with optional personalization.
    
    Without user profile: same as before (no personalization)
    With user profile:    adds user_tag_match signal (10% weight)
    """
    ce_normalized = max(0, min(1, (candidate['ce_score'] + 10) / 20))
    vote_normalized = np.log1p(candidate['score']) / np.log1p(26621)
    view_normalized = np.log1p(candidate['views']) / np.log1p(10000000)
    accepted_bonus = 1.0 if candidate['has_accepted'] else 0.0
    ans_normalized = np.log1p(candidate['answer_score']) / np.log1p(34269)
    
    try:
        year = pd.Timestamp(candidate['creation_date']).year
        freshness = max(0, min(1, (year - 2008) / (2024 - 2008)))
    except:
        freshness = 0.5
    
    if user_profile:
        # WITH personalization
        user_tag_match = compute_user_tag_match(candidate['tags'], user_profile)
        
        final_score = (
            0.35 * ce_normalized +
            0.15 * vote_normalized +
            0.15 * view_normalized +
            0.10 * accepted_bonus +
            0.05 * ans_normalized +
            0.05 * freshness +
            0.15 * user_tag_match       # PERSONALIZATION SIGNAL
        )
    else:
        # WITHOUT personalization (same as before)
        final_score = (
            0.40 * ce_normalized +
            0.20 * vote_normalized +
            0.15 * view_normalized +
            0.10 * accepted_bonus +
            0.10 * ans_normalized +
            0.05 * freshness
        )
    
    return final_score


def personalized_search(query, user_id=None, top_k=5):
    """
    Full personalized search pipeline.
    
    Stage 1: FAISS retrieves top 50 candidates
    Stage 2: Cross-Encoder re-ranks to top 20
    Stage 3: Weighted scoring with user personalization → top K
    
    If user_id is None or user has no history: runs without personalization.
    """
    # Get user profile (if user exists and has history)
    user_profile = None
    if user_id is not None:
        user_profile = build_user_tag_profile(user_id)
    
    # Stage 1: FAISS retrieval
    query_vec = bi_encoder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    faiss_scores, indices = index.search(query_vec, 50)
    
    candidates = []
    for faiss_score, idx in zip(faiss_scores[0], indices[0]):
        q_id = int(question_ids[idx])
        details = get_question_details(q_id)
        if details:
            details['faiss_similarity'] = float(faiss_score)
            candidates.append(details)
    
    # Stage 2: Cross-Encoder re-ranking
    pairs = [[query, c['title']] for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    for i, c in enumerate(candidates):
        c['ce_score'] = float(ce_scores[i])
    
    # Take top 20 by CE score
    candidates.sort(key=lambda x: x['ce_score'], reverse=True)
    candidates = candidates[:20]
    
    # Stage 3: Weighted scoring with personalization
    for c in candidates:
        c['final_score'] = compute_final_score(c, user_profile)
        if user_profile:
            c['tag_match'] = compute_user_tag_match(c['tags'], user_profile)
        else:
            c['tag_match'] = 0.0
    
    candidates.sort(key=lambda x: x['final_score'], reverse=True)
    
    # Log this search if user_id provided
    if user_id is not None:
        log_search(user_id, query)
    
    return candidates[:top_k], user_profile


print('Personalized search function ready!')

Personalized search function ready!


---
## 7. Cold Start Feed (Homepage Recommendations)

When a user opens the app, they see a homepage. What we show depends on whether they have history.

In [20]:
def get_homepage_feed(user_id=None, num_results=10):
    """
    Generate homepage recommendations.
    
    For NEW users (no click history):
      → Rule-based: trending + popular questions across diverse tags
    
    For RETURNING users (have click history):
      → Content-based: questions matching their tag preferences
    """
    user_profile = None
    feed_type = 'cold_start'
    
    if user_id is not None:
        user_profile = build_user_tag_profile(user_id)
    
    if user_profile:
        # PERSONALIZED FEED — user has history
        feed_type = 'personalized'
        
        # Get user's top 5 tags
        top_tags = list(user_profile.keys())[:5]
        
        # Find high-quality questions matching user's preferred tags
        mask = questions_df['primary_tag'].isin(top_tags)
        user_questions = questions_df[mask].copy()
        
        # Score by quality + tag preference match
        user_questions['feed_score'] = (
            np.log1p(user_questions['score']) * 0.4 +
            np.log1p(user_questions['view_count']) * 0.3 +
            user_questions['has_accepted_answer'].astype(float) * 0.3
        )
        
        # Apply tag weight boost
        user_questions['tag_boost'] = user_questions['primary_tag'].map(
            lambda t: user_profile.get(t, 0)
        )
        user_questions['feed_score'] = user_questions['feed_score'] * (1 + user_questions['tag_boost'])
        
        # Get top results, ensuring tag diversity
        results = []
        tags_used = set()
        
        for _, row in user_questions.nlargest(num_results * 3, 'feed_score').iterrows():
            if len(results) >= num_results:
                break
            # Allow max 3 questions per tag for diversity
            tag = row['primary_tag']
            tag_count = sum(1 for r in results if r.get('primary_tag') == tag)
            if tag_count < 3:
                details = get_question_details(int(row['id']))
                if details:
                    details['feed_reason'] = f"Because you like {tag}"
                    results.append(details)
    
    else:
        # COLD START FEED — no history, show popular + diverse
        results = []
        
        # Get top 2 questions from each of the top 5 tags (diversity)
        top_tags = questions_df['primary_tag'].value_counts().head(5).index.tolist()
        
        for tag in top_tags:
            tag_questions = questions_df[questions_df['primary_tag'] == tag]
            top_2 = tag_questions.nlargest(2, 'score')
            for _, row in top_2.iterrows():
                details = get_question_details(int(row['id']))
                if details:
                    details['feed_reason'] = f"Popular in {tag}"
                    results.append(details)
        
        results = results[:num_results]
    
    return results, feed_type, user_profile


print('Homepage feed function ready!')

Homepage feed function ready!


---
## 8. Display Helpers

In [23]:
def display_search_results(query, results, user_profile=None):
    """Pretty-print search results with personalization info."""
    print(f'\nQuery: "{query}"')
    
    if user_profile:
        top_prefs = list(user_profile.items())[:5]
        prefs_str = ', '.join(f'{tag}({w:.0%})' for tag, w in top_prefs)
        print(f'User preferences: {prefs_str}')
    else:
        print(f'User preferences: None (no personalization)')
    
    print('=' * 80)
    
    for i, r in enumerate(results, 1):
        accepted = ' [ACC]' if r.get('answer_accepted') or r.get('has_accepted') else ''
        tag_match_str = f'  tag_match: {r["tag_match"]:.2f}' if r.get('tag_match', 0) > 0 else ''
        
        print(f'\n  #{i} (score: {r["final_score"]:.3f}{tag_match_str})')
        print(f'  {r["title"]}')
        print(f'     Tags: {r["tags"]}  |  Votes: {r["score"]:,}  |  Views: {r["views"]:,}{accepted}')
    
    print('\n' + '=' * 80)


def display_feed(results, feed_type, user_profile=None):
    """Pretty-print homepage feed."""
    if feed_type == 'personalized':
        top_prefs = list(user_profile.items())[:5]
        prefs_str = ', '.join(f'{tag}({w:.0%})' for tag, w in top_prefs)
        print(f'\nHomepage Feed: PERSONALIZED')
        print(f'User preferences: {prefs_str}')
    else:
        print(f'\nHomepage Feed: COLD START (new user, no history)')
    
    print('=' * 80)
    
    for i, r in enumerate(results, 1):
        reason = r.get('feed_reason', '')
        print(f'  #{i}  {r["title"][:65]}')
        print(f'       Tags: {r["tags"]}  |  Votes: {r["score"]:,}  |  {reason}')
    
    print('\n' + '=' * 80)


print('Display functions ready!')

Display functions ready!


---
## 9. Demo — Simulate a Complete User Journey

We simulate two users with different interests and show how the system personalizes for each.

In [26]:
# Clean slate — reset database for fresh demo
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    init_database()

# Create two users
alice_id = create_user('alice')    # Will be a Python developer
bob_id = create_user('bob')        # Will be a JavaScript developer
print()

Database initialized: ../Dataset_Cleaned\recsys_users.db
  Created user: alice (id: 1)
  Created user: bob (id: 2)



In [28]:
# ============================================================
# STEP 1: Cold Start — both users are new, no history
# ============================================================
print('STEP 1: COLD START')
print('Both Alice and Bob are new users with no history.\n')

# Homepage feed for a new user
feed_results, feed_type, profile = get_homepage_feed(user_id=alice_id)
display_feed(feed_results, feed_type, profile)

STEP 1: COLD START
Both Alice and Bob are new users with no history.


Homepage Feed: COLD START (new user, no history)
  #1  How can I remove a specific item from an array?
       Tags: javascript,arrays  |  Votes: 10,953  |  Popular in javascript
  #2  How do I check if an element is hidden in jQuery?
       Tags: javascript,jquery,dom,visibility  |  Votes: 8,447  |  Popular in javascript
  #3  What does the "yield" keyword do?
       Tags: python,iterator,generator  |  Votes: 12,259  |  Popular in python
  #4  What does if __name__ == "__main__": do?
       Tags: python,namespaces,program-entry-point,python-module,idioms  |  Votes: 7,713  |  Popular in python
  #5  Why is processing a sorted array faster than processing an unsort
       Tags: java,c++,performance,cpu-architecture,branch-prediction  |  Votes: 26,621  |  Popular in java
  #6  Is Java "pass-by-reference" or "pass-by-value"?
       Tags: java,methods,parameter-passing,pass-by-reference,pass-by-value  |  Votes: 7,526  | 

In [30]:
# ============================================================
# STEP 2: Both search "how to handle errors" — SAME results
# ============================================================
print('STEP 2: SAME QUERY, NO HISTORY YET')
print('Both users search for the same thing. Results should be identical.\n')

alice_results, alice_profile = personalized_search('how to handle errors', user_id=alice_id)
print('--- ALICE ---')
display_search_results('how to handle errors', alice_results, alice_profile)

bob_results, bob_profile = personalized_search('how to handle errors', user_id=bob_id)
print('\n--- BOB ---')
display_search_results('how to handle errors', bob_results, bob_profile)

STEP 2: SAME QUERY, NO HISTORY YET
Both users search for the same thing. Results should be identical.

--- ALICE ---

Query: "how to handle errors"
User preferences: None (no personalization)

  #1 (score: 0.609)
  Handling specific errors in JavaScript (think exceptions)
     Tags: javascript,error-handling  |  Votes: 150  |  Views: 100,748 [ACC]

  #2 (score: 0.591)
  Best way to handle errors on a php page?
     Tags: php,exception,exception-handling,error-handling,try-catch  |  Votes: 37  |  Views: 19,854 [ACC]

  #3 (score: 0.587)
  How to Handle Errors in Node.js using Express
     Tags: node.js,error-handling,express  |  Votes: 12  |  Views: 16,781 [ACC]

  #4 (score: 0.577)
  How do I handle errors with promises?
     Tags: javascript,node.js,error-handling,promise,bluebird  |  Votes: 30  |  Views: 58,538 [ACC]

  #5 (score: 0.568)
  Error-Handling in Swift-Language
     Tags: swift,error-handling  |  Votes: 194  |  Views: 93,767 [ACC]


--- BOB ---

Query: "how to handle error

In [32]:
# ============================================================
# STEP 3: Simulate different browsing histories
# ============================================================
print('STEP 3: BUILDING USER HISTORIES')
print('Alice browses Python questions, Bob browses JavaScript questions.\n')

# Alice clicks on Python-related questions
alice_clicks = [
    (419163, 'python,namespaces,main', 7713),
    (403211, 'python,sorting,dictionary', 4256),
    (100003, 'python,oop,object', 6956),
    (38987,  'python,dictionary', 6456),
    (89228,  'python,shell,terminal', 5790),
    (273192, 'python,exception,error-handling', 5316),
    (394809, 'python,operators', 7477),
]

print('Alice clicks (Python developer):')
for q_id, tags, score in alice_clicks:
    log_click(alice_id, q_id, tags, score)
    print(f'  Clicked: tags={tags}')

# Bob clicks on JavaScript-related questions
bob_clicks = [
    (5767325, 'javascript,arrays', 10953),
    (208105,  'javascript,arrays,object', 7138),
    (1335851, 'javascript,jquery', 8264),
    (503093,  'javascript,html,dom', 7707),
    (111102,  'javascript,jquery,events', 7623),
    (336859,  'javascript,function,scope', 7477),
    (14220321,'javascript,node.js,async', 6442),
]

print('\nBob clicks (JavaScript developer):')
for q_id, tags, score in bob_clicks:
    log_click(bob_id, q_id, tags, score)
    print(f'  Clicked: tags={tags}')

# Show profiles
alice_profile = build_user_tag_profile(alice_id)
bob_profile = build_user_tag_profile(bob_id)

print(f'\nAlice tag profile: {dict(list(alice_profile.items())[:6])}')
print(f'Bob tag profile:   {dict(list(bob_profile.items())[:6])}')

STEP 3: BUILDING USER HISTORIES
Alice browses Python questions, Bob browses JavaScript questions.

Alice clicks (Python developer):
  Clicked: tags=python,namespaces,main
  Clicked: tags=python,sorting,dictionary
  Clicked: tags=python,oop,object
  Clicked: tags=python,dictionary
  Clicked: tags=python,shell,terminal
  Clicked: tags=python,exception,error-handling
  Clicked: tags=python,operators

Bob clicks (JavaScript developer):
  Clicked: tags=javascript,arrays
  Clicked: tags=javascript,arrays,object
  Clicked: tags=javascript,jquery
  Clicked: tags=javascript,html,dom
  Clicked: tags=javascript,jquery,events
  Clicked: tags=javascript,function,scope
  Clicked: tags=javascript,node.js,async

Alice tag profile: {'python': 0.3603, 'dictionary': 0.1176, 'namespaces': 0.0735, 'main': 0.0735, 'sorting': 0.0662, 'oop': 0.0588}
Bob tag profile:   {'javascript': 0.3798, 'arrays': 0.1473, 'jquery': 0.1085, 'object': 0.0698, 'html': 0.0543, 'dom': 0.0543}


In [34]:
# ============================================================
# STEP 4: SAME query, DIFFERENT results — THIS IS PERSONALIZATION
# ============================================================
print('STEP 4: SAME QUERY, PERSONALIZED RESULTS')
print('Both search "how to handle errors" again.')
print('Now they have different histories — results should differ.\n')

alice_results, alice_profile = personalized_search('how to handle errors', user_id=alice_id)
print('--- ALICE (Python developer) ---')
display_search_results('how to handle errors', alice_results, alice_profile)

bob_results, bob_profile = personalized_search('how to handle errors', user_id=bob_id)
print('\n--- BOB (JavaScript developer) ---')
display_search_results('how to handle errors', bob_results, bob_profile)

STEP 4: SAME QUERY, PERSONALIZED RESULTS
Both search "how to handle errors" again.
Now they have different histories — results should differ.

--- ALICE (Python developer) ---

Query: "how to handle errors"
User preferences: python(36%), dictionary(12%), namespaces(7%), main(7%), sorting(7%)

  #1 (score: 0.535  tag_match: 0.36)
  Error handling in SQLAlchemy
     Tags: python,sqlalchemy  |  Votes: 69  |  Views: 87,636 [ACC]

  #2 (score: 0.533  tag_match: 0.04)
  Handling specific errors in JavaScript (think exceptions)
     Tags: javascript,error-handling  |  Votes: 150  |  Views: 100,748 [ACC]

  #3 (score: 0.532  tag_match: 0.07)
  Best way to handle errors on a php page?
     Tags: php,exception,exception-handling,error-handling,try-catch  |  Votes: 37  |  Views: 19,854 [ACC]

  #4 (score: 0.529  tag_match: 0.04)
  How to Handle Errors in Node.js using Express
     Tags: node.js,error-handling,express  |  Votes: 12  |  Views: 16,781 [ACC]

  #5 (score: 0.516  tag_match: 0.04)
  Ho

In [36]:
# ============================================================
# STEP 5: Personalized Homepage Feeds
# ============================================================
print('STEP 5: PERSONALIZED HOMEPAGE FEEDS')
print('What each user sees when they open the app.\n')

print('--- ALICE\'s Homepage ---')
feed, feed_type, profile = get_homepage_feed(user_id=alice_id)
display_feed(feed, feed_type, profile)

print('\n--- BOB\'s Homepage ---')
feed, feed_type, profile = get_homepage_feed(user_id=bob_id)
display_feed(feed, feed_type, profile)

STEP 5: PERSONALIZED HOMEPAGE FEEDS
What each user sees when they open the app.

--- ALICE's Homepage ---

Homepage Feed: PERSONALIZED
User preferences: python(36%), dictionary(12%), namespaces(7%), main(7%), sorting(7%)
  #1  What does the "yield" keyword do?
       Tags: python,iterator,generator  |  Votes: 12,259  |  Because you like python
  #2  What does if __name__ == "__main__": do?
       Tags: python,namespaces,program-entry-point,python-module,idioms  |  Votes: 7,713  |  Because you like python
  #3  How do I execute a program or call a system command?
       Tags: python,shell,terminal,subprocess,command  |  Votes: 5,790  |  Because you like python


--- BOB's Homepage ---

Homepage Feed: PERSONALIZED
User preferences: javascript(38%), arrays(15%), jquery(11%), object(7%), html(5%)
  #1  How can I remove a specific item from an array?
       Tags: javascript,arrays  |  Votes: 10,953  |  Because you like javascript
  #2  How do I redirect to another webpage?
       Tags: java

In [38]:
# ============================================================
# STEP 6: Another query — "what is a closure"
# ============================================================
print('STEP 6: ANOTHER PERSONALIZED QUERY')
print('Both search "what is a closure" — a concept that exists in many languages.\n')

alice_results, alice_profile = personalized_search('what is a closure', user_id=alice_id)
print('--- ALICE (expects Python closures) ---')
display_search_results('what is a closure', alice_results, alice_profile)

bob_results, bob_profile = personalized_search('what is a closure', user_id=bob_id)
print('\n--- BOB (expects JavaScript closures) ---')
display_search_results('what is a closure', bob_results, bob_profile)

STEP 6: ANOTHER PERSONALIZED QUERY
Both search "what is a closure" — a concept that exists in many languages.

--- ALICE (expects Python closures) ---

Query: "what is a closure"
User preferences: python(36%), dictionary(12%), namespaces(7%), main(7%), sorting(7%)

  #1 (score: 0.521)
  What is a practical use for a closure in JavaScript?
     Tags: javascript,closures,terminology  |  Votes: 340  |  Views: 138,882 [ACC]

  #2 (score: 0.520)
  What are 'closures' in .NET?
     Tags: .net,closures  |  Votes: 211  |  Views: 45,952 [ACC]

  #3 (score: 0.509)
  Closure in Java 7
     Tags: java,closures  |  Votes: 106  |  Views: 126,054 [ACC]

  #4 (score: 0.509)
  What is Closure in Laravel?
     Tags: php,laravel,laravel-5,closures  |  Votes: 40  |  Views: 50,986 [ACC]

  #5 (score: 0.507)
  What exactly does "closure" refer to in JavaScript?
     Tags: javascript,closures,definition  |  Votes: 72  |  Views: 16,929 [ACC]


--- BOB (expects JavaScript closures) ---

Query: "what is a closu

---
## 10. Compare: With vs Without Personalization

In [41]:
# Show the difference personalization makes for a specific query
query = 'how to read a file'

print(f'Query: "{query}"')
print('=' * 85)

# Without personalization
results_no_personal, _ = personalized_search(query, user_id=None)
print('\n--- WITHOUT Personalization (anonymous user) ---')
for i, r in enumerate(results_no_personal, 1):
    print(f'  #{i}  [{r["tags"][:30]:<30s}]  Votes: {r["score"]:>5,}  {r["title"][:50]}')

# With Alice's profile
results_alice, profile_a = personalized_search(query, user_id=alice_id)
print('\n--- ALICE (Python developer, tag_match active) ---')
for i, r in enumerate(results_alice, 1):
    match = f'match:{r["tag_match"]:.2f}' if r['tag_match'] > 0 else 'no match'
    print(f'  #{i}  [{r["tags"][:30]:<30s}]  Votes: {r["score"]:>5,}  {match:<12s}  {r["title"][:45]}')

# With Bob's profile
results_bob, profile_b = personalized_search(query, user_id=bob_id)
print('\n--- BOB (JavaScript developer, tag_match active) ---')
for i, r in enumerate(results_bob, 1):
    match = f'match:{r["tag_match"]:.2f}' if r['tag_match'] > 0 else 'no match'
    print(f'  #{i}  [{r["tags"][:30]:<30s}]  Votes: {r["score"]:>5,}  {match:<12s}  {r["title"][:45]}')

print('\n' + '=' * 85)

Query: "how to read a file"

--- WITHOUT Personalization (anonymous user) ---
  #1  [c++,file-io,ofstream          ]  Votes:   747  Read file line by line using ifstream in C++
  #2  [java,file,file-io             ]  Votes:   279  What is simplest way to read a file into String?
  #3  [c++,iostream,fstream,file-hand]  Votes:    98  How to read a file line by line or a whole text fi
  #4  [java,arraylist,file-io,text-fi]  Votes:    81  Java: How to read a text file
  #5  [android,text-files,java-io    ]  Votes:   117  Reading a simple text file

--- ALICE (Python developer, tag_match active) ---
  #1  [c++,file-io,ofstream          ]  Votes:   747  no match      Read file line by line using ifstream in C++
  #2  [python                        ]  Votes:    33  match:0.36    How to read a file in other directory in pyth
  #3  [c++,iostream,fstream,file-hand]  Votes:    98  no match      How to read a file line by line or a whole te
  #4  [java,file,file-io             ]  Votes:   279  no 

---
## 11. Database Stats

In [44]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('SELECT COUNT(*) FROM users')
num_users = cursor.fetchone()[0]

cursor.execute('SELECT COUNT(*) FROM search_history')
num_searches = cursor.fetchone()[0]

cursor.execute('SELECT COUNT(*) FROM click_history')
num_clicks = cursor.fetchone()[0]

db_size = os.path.getsize(DB_PATH) / 1024

conn.close()

print('Database Stats:')
print(f'  Users:     {num_users}')
print(f'  Searches:  {num_searches}')
print(f'  Clicks:    {num_clicks}')
print(f'  DB size:   {db_size:.1f} KB')
print(f'  DB file:   {DB_PATH}')

Database Stats:
  Users:     2
  Searches:  8
  Clicks:    14
  DB size:   24.0 KB
  DB file:   ../Dataset_Cleaned\recsys_users.db


In [48]:
print('=' * 65)
print('PERSONALIZATION COMPLETE')
print('=' * 65)
print(f'\n  Type: Content-Based Filtering')
print(f'  Storage: SQLite (single file, no server)')
print(f'  Signal: user_tag_match (15% weight in scoring)')
print(f'\n  User Journey:')
print(f'    New user    → Cold start feed (trending + popular)')
print(f'    Few clicks  → Tag profile built, personalization starts')
print(f'    Active user → Full personalized search + homepage feed')
print(f'\n  Demo Results:')
print(f'    Alice (Python dev):     Python results boosted')
print(f'    Bob (JavaScript dev):   JavaScript results boosted')
print(f'    Same query, different results → Personalization works')
print(f'\n  Full Pipeline:')
print(f'    FAISS (retrieve 50) → Cross-Encoder (rank 20)')
print(f'    → Weighted Score + user_tag_match (top 10)')
print(f'\n  Files:')
print(f'    recsys_users.db     (SQLite, deploys with the app)')
print(f'    + all previous files (FAISS index, embeddings, data)')
print('=' * 65)

PERSONALIZATION COMPLETE

  Type: Content-Based Filtering
  Storage: SQLite (single file, no server)
  Signal: user_tag_match (15% weight in scoring)

  User Journey:
    New user    → Cold start feed (trending + popular)
    Few clicks  → Tag profile built, personalization starts
    Active user → Full personalized search + homepage feed

  Demo Results:
    Alice (Python dev):     Python results boosted
    Bob (JavaScript dev):   JavaScript results boosted
    Same query, different results → Personalization works

  Full Pipeline:
    FAISS (retrieve 50) → Cross-Encoder (rank 20)
    → Weighted Score + user_tag_match (top 10)

  Files:
    recsys_users.db     (SQLite, deploys with the app)
    + all previous files (FAISS index, embeddings, data)
